In [2]:
bronze_path = (
    "Files/bronze/open_meteo/forecast/"
    "edinburgh/ingestion_date=2026-11-07/response.json"
)

bronze_df = (
    spark.read
    .option("multiLine", "true")
    .json(bronze_path)
)

bronze_df.printSchema()

bronze_df.selectExpr(
    "size(hourly.time) AS hourly_records"
).show()

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 5, Finished, Available, Finished, False)

root
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- hourly: struct (nullable = true)
 |    |-- relative_humidity_2m: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- temperature_2m: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |-- hourly_units: struct (nullable = true)
 |    |-- relative_humidity_2m: string (nullable = true)
 |    |-- temperature_2m: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- utc_offset_seconds: long (nullable = true)

+--------------+
|hourly_records|
+--------------+
|            48|
+--------------+



In [3]:
from pyspark.sql import functions as F

bronze_df.select(
    F.size("hourly.time").alias("time_count"),
    F.size("hourly.temperature_2m").alias("temperature_count"),
    F.size("hourly.relative_humidity_2m").alias("humidity_count")
).show()

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 6, Finished, Available, Finished, False)

+----------+-----------------+--------------+
|time_count|temperature_count|humidity_count|
+----------+-----------------+--------------+
|        48|               48|            48|
+----------+-----------------+--------------+



In [4]:
from pyspark.sql import functions as F

zipped_df = bronze_df.select(
    F.explode(
        F.arrays_zip(
            F.col("hourly.time"),
            F.col("hourly.temperature_2m"),
            F.col("hourly.relative_humidity_2m")
        )
    ).alias("reading")
)

zipped_df.show(5, truncate= False)

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 7, Finished, Available, Finished, False)

+---------------------------+
|reading                    |
+---------------------------+
|{2026-11-07T00:00, 9.0, 85}|
|{2026-11-07T01:00, 9.3, 83}|
|{2026-11-07T02:00, 9.2, 84}|
|{2026-11-07T03:00, 9.0, 87}|
|{2026-11-07T04:00, 9.0, 87}|
+---------------------------+
only showing top 5 rows


In [5]:
from pyspark.sql import functions as F 

structured_df = zipped_df.select(
    F.to_timestamp(
        F.col("reading.time"),
        "yyyy-MM-dd'T'HH:mm"
    ).alias("event_time"),

    F.col("reading.temperature_2m")
        .cast("double")
        .alias("temperature_c"),

    F.col("reading.relative_humidity_2m")
        .cast("int")
        .alias("humidity_pct"),

    F.lit("edinburgh").alias("city"),

    F.lit("2026-11-07")
        .cast("date")
        .alias("ingestion_date")
)

structured_df.printSchema()
structured_df.show(5, truncate= False)

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 8, Finished, Available, Finished, False)

root
 |-- event_time: timestamp (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidity_pct: integer (nullable = true)
 |-- city: string (nullable = false)
 |-- ingestion_date: date (nullable = true)

+-------------------+-------------+------------+---------+--------------+
|event_time         |temperature_c|humidity_pct|city     |ingestion_date|
+-------------------+-------------+------------+---------+--------------+
|2026-11-07 00:00:00|9.0          |85          |edinburgh|2026-11-07    |
|2026-11-07 01:00:00|9.3          |83          |edinburgh|2026-11-07    |
|2026-11-07 02:00:00|9.2          |84          |edinburgh|2026-11-07    |
|2026-11-07 03:00:00|9.0          |87          |edinburgh|2026-11-07    |
|2026-11-07 04:00:00|9.0          |87          |edinburgh|2026-11-07    |
+-------------------+-------------+------------+---------+--------------+
only showing top 5 rows


In [6]:
#Identify records that violate our quality rules
invalid_df = structured_df.filter(
    F.col("event_time").isNull()
    | F.col("city").isNull()
    | (
        F.col("humidity_pct").isNotNull()
        & ~F.col("humidity_pct").between(0, 100)
    )
)

#Check duplicates business keys
duplicate_df = (
    structured_df.groupBy("city", "event_time")
    .count()
    .filter(F.col("count")>1)
)

print("Total records:", structured_df.count())
print("invalid record:", invalid_df.count())
print("duplicate business keys:", duplicate_df.count())


StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 9, Finished, Available, Finished, False)

Total records: 48
invalid record: 0
duplicate business keys: 0


In [7]:
# Stop the pipeline if invalid records exist
invalid_count = invalid_df.count()

if invalid_count > 0:
    invalid_df.show(truncate=False)
    raise ValueError(
        f"Data quality check failed: {invalid_count} invalid records"
    )

# Remove completely identical records
clean_df = structured_df.dropDuplicates()

# Check for conflicting records with the same business key
conflicting_keys = (
    clean_df
    .groupBy("city", "event_time")
    .count()
    .filter(F.col("count") > 1)
)

if conflicting_keys.count() > 0:
    conflicting_keys.show()
    raise ValueError("Conflicting forecasts found for the same city and hour")

print("Records ready for Silver:", clean_df.count())

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 10, Finished, Available, Finished, False)

Records ready for Silver: 48


In [8]:

from delta.tables import DeltaTable

table_name = "dbo.weather_forecast_silver"

if not spark.catalog.tableExists(table_name):

    # First run: create the Delta table
    (
        clean_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(table_name)
    )

    print("Silver Delta table created")

else:

    # Subsequent runs: update existing or insert new records
    target = DeltaTable.forName(spark, table_name)

    (
        target.alias("t")
        .merge(
            clean_df.alias("s"),
            "t.city = s.city AND t.event_time = s.event_time"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Silver Delta MERGE completed")


StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 11, Finished, Available, Finished, False)

Silver Delta MERGE completed


In [9]:
spark.sql("""
    SELECT
        city,
        COUNT(*) AS total_records,
        COUNT(DISTINCT event_time) as unique_hours
    FROM dbo.weather_forecast_silver
    GROUP BY city

""").show()

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 12, Finished, Available, Finished, False)

+---------+-------------+------------+
|     city|total_records|unique_hours|
+---------+-------------+------------+
|edinburgh|           48|          48|
+---------+-------------+------------+



In [10]:
spark.sql("""
    SELECT
    city,
    event_time,
    COUNT(*) AS record_count
    FROM dbo.weather_forecast_silver
    GROUP BY city, event_time
    HAVING COUNT(*) >1
    """
).show()

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 13, Finished, Available, Finished, False)

+----+----------+------------+
|city|event_time|record_count|
+----+----------+------------+
+----+----------+------------+



## Delta Lake MERGE — Update and Insert Test

### Objective
Validate that the Silver Delta table supports updates and inserts without creating duplicate records.

### Test environment
A separate test table, `dbo.weather_forecast_silver_merge_test`, is used to avoid modifying the main Silver table.

### Test scenarios
1. **UPDATE:** Increase an existing hourly temperature by 1°C and verify that MERGE updates the matching record.
2. **INSERT:** Introduce a new hourly record and verify that MERGE inserts it.
3. **DEDUPLICATION:** Confirm that each combination of `city` and `event_time` remains unique.

### Expected results
- The existing temperature changes from 9.0°C to 10.0°C.
- The test table increases from 48 to 49 records.
- No duplicate `(city, event_time)` keys exist.
- The original Silver table remains unchanged.

### Engineering concepts demonstrated
Delta Lake MERGE, upsert operations, composite business keys, idempotency and isolated testing.

**Note:** This section is for learning and validation. It is not part of the production ingestion workflow.


In [11]:
test_table = "dbo.weather_forecast_silver_merge_test"

(
    clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(test_table)
)

print("Test table created with:", spark.table(test_table).count(), "records")

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 14, Finished, Available, Finished, False)

Test table created with: 48 records


In [12]:
from pyspark.sql import functions as F

# Take an existing forecast record
existing_df = clean_df.orderBy("event_time").limit(1)

# Simulate a temperature revision of +1 degree
updated_df = existing_df.withColumn(
    "temperature_c",
    F.coalesce(F.col("temperature_c"), F.lit(0.0)) + 1.0
)

# Take the latest forecast and simulate a new forecast hour
new_df = (
    clean_df
    .orderBy(F.desc("event_time"))
    .limit(1)
    .withColumn(
        "event_time",
        F.expr("event_time + INTERVAL 1 HOUR")
    )
)

# Combine both records into an incoming batch
incoming_df = updated_df.unionByName(new_df)

incoming_df.select(
    "city", "event_time", "temperature_c"
).show(truncate=False)

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 15, Finished, Available, Finished, False)

+---------+-------------------+-------------+
|city     |event_time         |temperature_c|
+---------+-------------------+-------------+
|edinburgh|2026-11-07 00:00:00|10.0         |
|edinburgh|2026-11-09 00:00:00|7.5          |
+---------+-------------------+-------------+



In [13]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, test_table)

(
    target.alias("t")
    .merge(
        incoming_df.alias("s"),
        "t.city = s.city AND t.event_time = s.event_time"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 16, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [15]:
spark.sql(f"""
    SELECT
        city,
        COUNT(*) AS total_records,
        COUNT(DISTINCT event_time) AS unique_hours
    FROM {test_table}
    GROUP BY city
""").show()

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 19, Finished, Available, Finished, False)

+---------+-------------+------------+
|     city|total_records|unique_hours|
+---------+-------------+------------+
|edinburgh|           49|          49|
+---------+-------------+------------+



In [16]:
original_row = existing_df.first()
spark.table(test_table).filter(
    (F.col("city") == original_row["city"])
    & (F.col("event_time") == original_row["event_time"])
).select("city", "event_time", "temperature_c").show(truncate=False)

print("Original temperature:", original_row["temperature_c"])

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 20, Finished, Available, Finished, False)

+---------+-------------------+-------------+
|city     |event_time         |temperature_c|
+---------+-------------------+-------------+
|edinburgh|2026-11-07 00:00:00|10.0         |
+---------+-------------------+-------------+

Original temperature: 9.0


In [14]:
print(notebookutils.lakehouse.get().displayName)

StatementMeta(, d479247c-a1fb-4f96-b70b-7769150fe0c8, 17, Finished, Available, Finished, False)

windrose_lh
